### Connexion à la DB DuckDB

In [1]:
import duckdb
import os
from pathlib import Path
from typing import List
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np
from tqdm import tqdm
import sklearn
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

### Connexion à la DB / Import des Data


In [2]:
# Store database at project root
DB_NAME = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/amazing.duckdb") 
# Go up one level from current directory to get to project root
data_folder = Path("..") / "data"
# For absolute certainty, you could use the absolute path
# data_folder = Path("/home/c-enjalbert/Documents/EPSI/MSPR/bloc_2/amazing/data")
con = duckdb.connect(str(DB_NAME))

In [3]:
# 2. Query to list all tables in the database
# DuckDB specific way to list tables
tables_info = con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'main'
    ORDER BY table_name
""").df()

print(f"Found {len(tables_info)} tables in the database:\n")

if len(tables_info) > 0:
    for i, table_name in enumerate(tables_info['table_name']):
        print(f"{i+1}. {table_name}")
else:
    print("No tables found in the database.")

Found 3 tables in the database:

1. all_events
2. loaded_files
3. user_events


In [4]:
# 5. Alternative way to show all tables
print("List of all tables using DuckDB's connections.tables():")
con.sql("SHOW TABLES").show()

List of all tables using DuckDB's connections.tables():
┌──────────────┐
│     name     │
│   varchar    │
├──────────────┤
│ all_events   │
│ loaded_files │
│ user_events  │
└──────────────┘



In [5]:
# Examine the all_events table
print("First 10 rows of all_events table:")
all_events_data = con.sql("""
    SELECT * FROM all_events LIMIT 10
""")
all_events_data.show()


# Examine the loaded_files table
print("\nContents of loaded_files table:")
loaded_files_data = con.sql("""
    SELECT * FROM loaded_files
""")
loaded_files_data.show()


First 10 rows of all_events table:
┌─────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────────┬─────────┬─────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │         category_code          │  brand  │  price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │  varchar   │       varchar       │            varchar             │ varchar │ double  │  varchar  │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼────────────────────────────────┼─────────┼─────────┼───────────┼──────────────────────────────────────┤
│ 2019-12-01 00:00:00 │ view       │ 1005105    │ 2232732093077520756 │ construction.tools.light       │ apple   │ 1302.48 │ 556695836 │ ca5eefc5-11f9-450c-91ed-380285a0bc80 │
│ 2019-12-01 00:00:00 │ view       │ 22700068   │ 2232732091643068746 │ NULL         

In [6]:
# Examine the all_events table
print("First 10 rows of all_events table:")
all_events_data = con.sql("""
    SELECT * FROM all_events LIMIT 10
""")
all_events_data.show()

# Show count of records in all_events
record_count = con.sql("""
    SELECT COUNT(*) as total_events FROM all_events
""")
record_count.show()

# Examine the loaded_files table
print("\nContents of loaded_files table:")
loaded_files_data = con.sql("""
    SELECT * FROM loaded_files
""")
loaded_files_data.show()


First 10 rows of all_events table:
┌─────────────────────┬────────────┬────────────┬─────────────────────┬────────────────────────────────┬─────────┬─────────┬───────────┬──────────────────────────────────────┐
│     event_time      │ event_type │ product_id │     category_id     │         category_code          │  brand  │  price  │  user_id  │             user_session             │
│      timestamp      │  varchar   │  varchar   │       varchar       │            varchar             │ varchar │ double  │  varchar  │               varchar                │
├─────────────────────┼────────────┼────────────┼─────────────────────┼────────────────────────────────┼─────────┼─────────┼───────────┼──────────────────────────────────────┤
│ 2019-12-01 00:00:00 │ view       │ 1005105    │ 2232732093077520756 │ construction.tools.light       │ apple   │ 1302.48 │ 556695836 │ ca5eefc5-11f9-450c-91ed-380285a0bc80 │
│ 2019-12-01 00:00:00 │ view       │ 22700068   │ 2232732091643068746 │ NULL         

In [7]:
DB_NAME = "amazing.duckdb"
TABLE_EVENTS = "all_events"
TABLE_USER_EVENTS = "user_events"
SAMPLE_USER_PERCENT = 0.005
BATCH_SIZE = 1000 

### Import de la table DuckDB

In [8]:
# Chargement de users avec au moins 10 événements 
print("Chargement d'un échantillon d'utilisateurs actifs...")

# Afficher les tables disponibles dans la base de données
tables_df = con.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'main'").fetch_df()
print("Tables disponibles dans la base de données :")
print(tables_df)

user_ids_df = con.execute(f"""
    SELECT user_id
    FROM all_events
    WHERE user_id IS NOT NULL
    GROUP BY user_id
    HAVING COUNT(*) >= 10
""").fetch_df()

Chargement d'un échantillon d'utilisateurs actifs...
Tables disponibles dans la base de données :
     table_name
0    all_events
1  loaded_files
2   user_events


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

### Normalisation et Standardisation des données

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

sampled_user_ids = user_ids_df.sample(frac=SAMPLE_USER_PERCENT, random_state=42)['user_id'].tolist()

print(f"Nombre d'utilisateurs actifs échantillonnés : {len(sampled_user_ids)}")

#  Création des features utilisateurs batch par batch 
print("Création des features utilisateurs par batch...")

user_features_list = []

for i in tqdm(range(0, len(sampled_user_ids), BATCH_SIZE), desc="Avancement user features", ncols=100):
    batch_ids = sampled_user_ids[i:i+BATCH_SIZE]
    batch_ids_str = ",".join(f"'{uid}'" for uid in batch_ids)

    batch_query = f"""
    WITH
        base_events AS (
            SELECT
                user_id,
                event_type,
                event_time,
                price,
                category_code,
                LEAD(event_time) OVER (PARTITION BY user_id ORDER BY event_time) AS next_event_time
            FROM {TABLE_EVENTS}
            WHERE user_id IN ({batch_ids_str})
        ),
        features AS (
            SELECT
                user_id,
                COUNT(*) AS total_events,
                SUM(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS total_views,
                SUM(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS total_purchases,
                AVG(EXTRACT(EPOCH FROM (next_event_time - event_time))) AS avg_time_between_events,
                SUM(CASE WHEN event_type = 'purchase' THEN price ELSE 0 END) AS total_spent,
                COALESCE(AVG(CASE WHEN event_type = 'purchase' THEN price ELSE NULL END), 0) AS avg_basket,
                MAX(event_time) AS last_event_time
            FROM base_events
            GROUP BY user_id
    )
    SELECT
        *,
        CASE WHEN total_views > 0 THEN total_purchases * 1.0 / total_views ELSE 0 END AS conversion_rate,
        CASE WHEN (total_views + total_purchases) > 0 THEN total_purchases * 1.0 / (total_views + total_purchases) ELSE 0 END AS purchase_ratio,
        DATE_PART('day', CAST('2020-03-31 22:00:00' AS TIMESTAMP) - last_event_time) AS days_since_last_event
    FROM features
    """

    batch_features = con.execute(batch_query).fetch_df()

    # Récupérer les user_id de cette batch
    valid_user_ids = con.execute(f"""
        SELECT user_id
        FROM {TABLE_EVENTS}
        WHERE user_id IN ({batch_ids_str})
        GROUP BY user_id
        HAVING COUNT(*) >= 10
    """).fetch_df()
    valid_user_ids = set(valid_user_ids["user_id"].astype(str))

    # Filtrage strict des user_id valides
    batch_features = batch_features[batch_features["user_id"].astype(str).isin(valid_user_ids)]

    # Compute category embeddings for the batch
    category_data = con.execute(f"""
        SELECT user_id, category_code
        FROM {TABLE_EVENTS}
        WHERE user_id IN ({batch_ids_str})
        AND category_code IS NOT NULL
    """).fetch_df()

    if not category_data.empty:
        user_categories = category_data.groupby("user_id")["category_code"].apply(list)

        def expand_hierarchy(categories):
            expanded = []
            for category in categories:
                parts = category.split(".")
                expanded.extend([".".join(parts[:i+1]) for i in range(len(parts))])
            return expanded

        user_hierarchy = user_categories.apply(expand_hierarchy)
        user_tokens = user_hierarchy.apply(lambda x: " ".join(x))

        vectorizer = TfidfVectorizer()
        tfidf_matrix = vectorizer.fit_transform(user_tokens)

        svd = TruncatedSVD(n_components=10, random_state=42)
        category_embeddings = svd.fit_transform(tfidf_matrix)

        category_embedding_df = pd.DataFrame(
            category_embeddings,
            index=user_tokens.index,
            columns=[f"category_emb_{i+1}" for i in range(category_embeddings.shape[1])]
        )

        # Add the embeddings as a single column
        category_embedding_df["category_embedding"] = category_embedding_df.values.tolist()

        # Merge embeddings with batch features
        batch_features = batch_features.merge(
            category_embedding_df[["category_embedding"]],
            left_on="user_id",
            right_index=True,
            how="left"
        )

    user_features_list.append(batch_features)

# Fusionner tous les batchs
user_features = pd.concat(user_features_list, ignore_index=True)

# Vérification des NaN
print("Vérification des NaN")
nan_summary = user_features.isna().sum()
print("Résumé des NaN par colonne :")
print(nan_summary[nan_summary > 0])

users_with_nan = user_features[user_features.isna().any(axis=1)]
print(f"Nombre d'utilisateurs avec des NaN : {len(users_with_nan)}")
print("Exemples d'utilisateurs avec NaN :")
print(users_with_nan.head(10))

Nombre d'utilisateurs actifs échantillonnés : 20837
Création des features utilisateurs par batch...


Avancement user features:  90%|█████████████████████████████████▍   | 19/21 [01:25<00:09,  4.52s/it]

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Avancement user features: 100%|█████████████████████████████████████| 21/21 [01:34<00:00,  4.51s/it]

Vérification des NaN
Résumé des NaN par colonne :
category_embedding    371
dtype: int64
Nombre d'utilisateurs avec des NaN : 371
Exemples d'utilisateurs avec NaN :
       user_id  total_events  total_views  total_purchases  \
108  530991189            13         13.0              0.0   
154  564044161            28         28.0              0.0   
176  568962380            25         25.0              0.0   
208  526679296            19         14.0              2.0   
231  558143838            80         74.0              3.0   
293  539549467            24         23.0              1.0   
316  571800530            22         20.0              0.0   
418  529710456            12         12.0              0.0   
425  585604954            15         13.0              0.0   
466  616091674            29         22.0              2.0   

     avg_time_between_events  total_spent  avg_basket     last_event_time  \
108            454160.416667         0.00       0.000 2020-02-28 12:38:34  

In [12]:
# Standardisation
print("Standardisation des features...")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(user_features.drop(columns=["last_event_time","category_embedding"]))

Standardisation des features...


### Feature Engineering - Category Embedding with TF-IDF

In [ ]:
# Compute category embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

# STEP 1: User → List of Purchased Categories
user_categories = user_features.groupby("user_id")["category_code"].apply(list)

# STEP 2: Hierarchy Expansions
def expand_hierarchy(categories):
    expanded = []
    for category in categories:
        parts = category.split(".")
        expanded.extend([".".join(parts[:i+1]) for i in range(len(parts))])
    return expanded

user_hierarchy = user_categories.apply(expand_hierarchy)

# STEP 3: User as “Document” of Tokens
user_tokens = user_hierarchy.apply(lambda x: " ".join(x))

# STEP 4: TF-IDF Vectorization
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(user_tokens)

# STEP 5: Dimensionality Reduction (SVD)
svd = TruncatedSVD(n_components=10, random_state=42)
category_embeddings = svd.fit_transform(tfidf_matrix)

# Create a DataFrame for embeddings
category_embedding_df = pd.DataFrame(
    category_embeddings,
    index=user_tokens.index,
    columns=[f"category_emb_{i+1}" for i in range(category_embeddings.shape[1])]
)

# Add the embeddings as a single column (optional: keep as separate columns if needed)
category_embedding_df["category_embedding"] = category_embedding_df.values.tolist()

# Verify the embeddings
print(category_embedding_df.head())

In [ ]:
user_features

,user_id,total_events,total_views,total_purchases,avg_time_between_events,total_spent,avg_basket,last_event_time,conversion_rate,purchase_ratio,days_since_last_event
0,577693164,13,11.0,0.0,323723.583333,0.00,0.00,2020-01-11 13:31:12,0.000000,0.000000,80
1,551647947,276,270.0,1.0,42848.098182,40.93,40.93,2020-02-23 12:41:34,0.003704,0.003690,37
2,560425126,14,14.0,0.0,339127.538462,0.00,0.00,2020-01-02 11:53:02,0.000000,0.000000,89
3,519244282,29,28.0,0.0,300658.714286,0.00,0.00,2020-02-29 02:58:10,0.000000,0.000000,31
4,591048134,14,14.0,0.0,94479.923077,0.00,0.00,2020-01-06 06:32:08,0.000000,0.000000,85
...,...,...,...,...,...,...,...,...,...,...,...
4167446,513038428,62,62.0,0.0,202458.655738,0.00,0.00,2020-02-26 05:30:03,0.000000,0.000000,34
4167447,547625068,12,12.0,0.0,531046.545455,0.00,0.00,2020-02-23 10:10:33,0.000000,0.000000,37
4167448,544788362,15,15.0,0.0,873848.428571,0.00,0.00,2020-02-26 07:20:14,0.000000,0.000000,34
4167449,536133904,20,17.0,2.0,210891.894737,687.20,343.60,2019-11-27 11:00:50,0.117647,0.105263,125


In [13]:
#  Sauvegarde des résultats dans DuckDB
print(f"Sauvegarde dans {TABLE_USER_EVENTS}...")
con.execute(f"DROP TABLE IF EXISTS {TABLE_USER_EVENTS}")
con.register("temp_user_features", user_features)
con.execute(f"CREATE TABLE {TABLE_USER_EVENTS} AS SELECT * FROM temp_user_features")

Sauvegarde dans user_events...


In [14]:
con.close()